# Dr. Max: teste de consistência v2
IP-Adapter **plus-face** (só o rosto), escala menor, prompt *solo* e 2 variações por cena.
Requer: **Accelerator = GPU T4** e **Internet = On**.

In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors

In [ ]:
import torch, os, shutil, time
from PIL import Image
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

MODEL = 'Lykon/dreamshaper-8'   # licença CreativeML OpenRAIL-M (permite uso comercial)
OUT = '/kaggle/working/cenas'
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT, exist_ok=True)

pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, algorithm_type='dpmsolver++', final_sigmas_type='sigma_min', use_karras_sigmas=True)
pipe = pipe.to('cuda')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
PERSONAGEM = 'solo, one man, friendly doctor, short dark brown hair, round glasses, white lab coat, light blue shirt, red tie, stethoscope'
ESTILO = '2d cartoon illustration, flat colors, bold outlines, cel shading, high quality'
NEGATIVO = ('two people, twins, duplicate, multiple people, crowd, photo, realistic, 3d render, deformed, '
            'bad anatomy, extra fingers, mutated hands, blurry, low quality, text, watermark, logo')

ref_prompt = f'{PERSONAGEM}, front view portrait, head and shoulders, smiling, plain light background, {ESTILO}'
refs = []
for s in range(4):
    g = torch.Generator('cuda').manual_seed(100 + s)
    img = pipe(ref_prompt, negative_prompt=NEGATIVO, width=512, height=512,
               num_inference_steps=28, guidance_scale=7, generator=g).images[0]
    img.save(f'{OUT}/ref_{s}.png'); refs.append(img)
sheet = Image.new('RGB', (2048, 512), 'white')
for i, r in enumerate(refs): sheet.paste(r, (i * 512, 0))
sheet.save(f'{OUT}/refs_opcoes.png')
sheet.resize((1024, 256))

In [ ]:
REF_INDEX = 0
ref = refs[REF_INDEX]
pipe.load_ip_adapter('h94/IP-Adapter', subfolder='models', weight_name='ip-adapter-plus-face_sd15.safetensors')
pipe.set_ip_adapter_scale(0.45)

cenas = [
    'medium shot, waving hello with one hand, standing in a bright clinic office',
    'wide shot, sitting on a sofa, tired and yawning, cozy living room, afternoon light',
    'close-up of face, explaining with index finger raised, clinic background',
    'medium shot, pointing at a big alarm clock on the wall, bedroom at night',
    'medium shot, smiling, thumbs up, sunny park with trees',
]
W, H = 768, 432
SEEDS = [42, 7]
grid = []
t0 = time.time()
for i, c in enumerate(cenas, 1):
    row = []
    for v, sd in enumerate(SEEDS):
        g = torch.Generator('cuda').manual_seed(sd)
        img = pipe(f'{PERSONAGEM}, {c}, {ESTILO}', negative_prompt=NEGATIVO, ip_adapter_image=ref,
                   width=W, height=H, num_inference_steps=28, guidance_scale=7.5, generator=g).images[0]
        img = img.resize((1536, 864), Image.LANCZOS)
        img.save(f'{OUT}/{i:03d}_{chr(97+v)}.png'); row.append(img)
    grid.append(row); print(f'cena {i:03d} ok')
print(f'{len(cenas)*len(SEEDS)} imagens em {time.time()-t0:.0f}s')

In [ ]:
tw, th = 384, 216
contato = Image.new('RGB', (th + tw * 2, th * len(grid)), 'white')
for r, row in enumerate(grid):
    contato.paste(ref.resize((th, th)), (0, r * th))
    for v, im in enumerate(row):
        contato.paste(im.resize((tw, th)), (th + v * tw, r * th))
contato.save('/kaggle/working/folha_contato.png')
shutil.make_archive('/kaggle/working/video_cenas', 'zip', OUT)
contato